# 攻撃的な投稿に対して落ち着いた返答を作る (カウンタースピーチ)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_dialogue/counter_speech.ipynb)

NLP クックブック 23-10 ／ 対話システム

差別的・攻撃的な投稿に対して、削除するだけでなく、冷静に反論したり、言われた側に寄り添ったりする返答を作りたい。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=140, **kwargs):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

posts = ["女に管理職なんて無理に決まってる。感情的だからな。", "外国人が増えたせいで治安が悪くなった。全員出て行ってほしい。",
         "あの人、日本語うまいね。外国人なのに。", "年寄りは新しいことを覚えられないんだから、さっさと引退すればいいのに。"]

STRATEGIES = {
    "事実で反論": "事実や根拠を示して、決めつけが正しくないことを穏やかに指摘する",
    "問いかけ": "相手を責めずに、なぜそう思うのかを問いかけて、考え直すきっかけを作る",
    "当事者への共感": "この投稿で傷つく人の立場に立ち、その人たちを支える言葉をかける",
}
PROMPT = ("SNS に次の投稿がありました。これに対する返信を書いてください。\n方針: {strategy}。\n"
          "条件: 相手を侮辱しない。攻撃的な言葉を使わない。2 文以内で、丁寧な言葉で書く。返信の文だけを出力する。\n\n投稿: {post}")
for post in posts[:2]:
    print("投稿:", post)
    for label, strategy in STRATEGIES.items():
        print(f"   [{label}] {chat(PROMPT.format(strategy=strategy, post=post))}")

投稿: 女に管理職なんて無理に決まってる。感情的だからな。
   [事実で反論] 管理職には論理的思考と問題解決能力が求められるため、性別だけで務まらないと判断するのは公平ではありません。感情のコントロールは個人次第であり、管理職に必要な資質は別にあります。
   [問いかけ] 女性が管理職に就くことに反対の意見ですね。確かに、感情は仕事においても重要な要素だと思います。しかし、管理職には論理的思考や問題解決能力も求められます。もしよろしければ、どのようにしたら女性がこれらの能力を発揮できるとお考えでしょうか？
   [当事者への共感] 女性だから管理職に適さないという考え方は間違っています。能力や適性は性別ではなく個人によって異なります。あなたの活躍を応援しています。
投稿: 外国人が増えたせいで治安が悪くなった。全員出て行ってほしい。
   [事実で反論] 外国人が増えたことが直ちに治安悪化につながるとは限りません。犯罪率のデータなど、具体的な根拠を示して議論することが大切です。お互いを尊重し、peacefulな解決策を見つけることが重要ですね。
   [問いかけ] 外国人が増えたことで感じる変化について、あなたは心配されているのですね。具体的にどのような点で治安が悪化したと感じていらっしゃいますか？もしよろしければ、詳細を教えていただけると、より建設的な議論ができるかもしれません。
   [当事者への共感] 外国人が増えたことで感じる不安について、共感します。しかし、多様性は私たちの地域社会を豊かにする要素でもあります。もし何か困ったことがあれば、いつでも助けますので、安心して生活していきましょう。


In [2]:
# はっきりした暴言ではないが、相手を傷つける発言 (マイクロアグレッション) への返答
for post in posts[2:]:
    print("投稿:", post)
    print("   ", chat(PROMPT.format(strategy="悪気がないかもしれないことを踏まえつつ、その言い方がなぜ相手を傷つけうるのかを、やさしく説明する", post=post)))

投稿: あの人、日本語うまいね。外国人なのに。
    その表現は、相手の努力や背景を尊重していないように感じられるかもしれません。外国語を上手に話すことは、その人の努力や能力を示すものであり、出身国だけで評価すべきではありません。相手を尊重するためには、その言語スキルそのものを褒めると良いでしょう。例えば、「その日本語、とても流暢で素敵ですね」といった表現がおすすめです。
投稿: 年寄りは新しいことを覚えられないんだから、さっさと引退すればいいのに。
    お年寄りの方々は長年の経験と知識をお持ちです。新しいことを学ぶ意欲も素晴らしいと思います。言い方には気をつけましょう。


In [3]:
# 生成した返答を機械的に点検する: 返答自体が攻撃的になっていないか、根拠のない数字や事実を作っていないか
import re

NG = ["バカ", "馬鹿", "アホ", "クズ", "黙れ", "お前", "頭が悪い", "恥を知れ", "差別主義者"]
def check(reply):
    return {"攻撃的な語": [w for w in NG if w in reply], "数字 (出典の確認が必要)": re.findall(r"\d+[%％割倍]|\d{4} 年", reply), "長さ": len(reply)}

for post in posts[:2]:
    reply = chat(PROMPT.format(strategy=STRATEGIES["事実で反論"], post=post))
    print(reply, "\n   →", check(reply))

管理職には論理的思考と問題解決能力が求められるため、性別だけで務まらないと判断するのは公平ではありません。感情のコントロールは個人次第であり、管理職に必要な資質は別にあります。 
   → {'攻撃的な語': [], '数字 (出典の確認が必要)': [], '長さ': 88}
外国人が増えたことが直ちに治安悪化につながるとは限りません。犯罪率のデータなど、具体的な根拠を示して議論することが大切です。お互いを尊重し、peacefulな解決策を見つけることが重要ですね。 
   → {'攻撃的な語': [], '数字 (出典の確認が必要)': [], '長さ': 96}


- カウンタースピーチは、有害な投稿を削除するのではなく、言葉で対抗する取り組み。事実を示す、矛盾を指摘する、問いかける、当事者に共感を示す、などの方針がある。どれが有効かは場面による。
- 明確な暴言だけでなく、悪意のない決めつけや「ほめ言葉」の形をとった偏見 (マイクロアグレッション) も対象になる。こちらは、相手を責めずに気付きを促す返答が向く。
- LLM は、もっともらしい統計や研究結果を作ってしまうことがある。事実で反論する返答は、根拠を人が確認してから使う。確認できない数字は入れさせない。
- 小さなモデルの出力には、実際に次のような失敗が出た: 英単語が混ざる (「peacefulな」)、当事者に寄り添う方針なのに、投稿者本人に向けて「応援しています」と書いてしまう、文の途中で切れる。方針が守られているかは、生成のたびに確かめる必要がある。
- 返答が説教調・上から目線になると逆効果になる。生成した返答が攻撃的でないかを点検し、最終的には人が読んでから投稿する。自動で返信する仕組みにはしない。
- モデルによっては、有害な投稿を入力に含むだけで応答を拒否することがある。その場合は、目的 (有害表現への対抗) を指示の中で明確にする。